# 04 — EXPLAIN sweep

Plans every `(SQL, params)` pair the three matrix reports recorded, in **both
index states**, and reports the contrast.

Companion to `03_api_index_matrix.ipynb`, which produced the reports. This
notebook does not drive APIs and does not need a Polaris restart — it replays
statements against PostgreSQL with `EXPLAIN`, never `EXPLAIN ANALYZE`.

**Why plain EXPLAIN.** It does not execute, which is what makes the write half
askable at all and leaves no clock to misquote. Plan *shape* is deterministic
here; execution time is not — the same plan measured 7.523 / 3.289 / 4.571 /
1.622 ms across repeats, a 4.6x spread that once moved a reported speedup from
11.6x to 6.3x on noise alone.

## What "index absent" means here

`idx_grant_records_grantee` is in **no schema file in this repo** — it is a
hand-created experimental index. So the index-absent state is not a degraded
one, it is **the stock Polaris schema**, and sweeping it alone is a complete
finding about what a real deployment does. The toggle answers a different and
optional question: *would* that index help.

## Run order

`Restart & Run All`. The notebook works from **whatever index state the cluster
is in**, sweeps that state, toggles, sweeps the other, and restores. It never
assumes it starts with the index present.

**Cell 2 arms the toggle.** With `ARM_TOGGLE = False` the notebook sweeps only
the current state and stops — safe, and useful on its own.


In [1]:
# --- cell 1: bootstrap ------------------------------------------------------
import gc
import json
import os
import pathlib
import shlex
import subprocess
import sys
from datetime import datetime

sys.path.insert(0, str(pathlib.Path.cwd().parent.parent / "src"))

from polaris_test_utils import *  # noqa: F403 — init_env, root_token, PG_* globals

init_env("local")
require_not_prod("04: drops and recreates an index on grant_records")

#: Second guard, independent of require_not_prod. This notebook DROPS an index
#: on the table the whole audit measures. A wrong endpoint here is not a bad
#: measurement, it is a production change.
assert any(h in POLARIS_URL for h in ("localhost", "127.0.0.1", "192.168.")), (
    f"Refusing to run against {POLARIS_URL} — 04 is local-only."
)

HERE = pathlib.Path.cwd()
RUNS = HERE / "runs"
REPORTS = HERE / "reports"
RUNS.mkdir(exist_ok=True)

#: Cluster coordinates. `init_env` supplies PG_USER / PG_PASSWORD / PG_DB but
#: NOT these — they are kubectl's, not the database's. Same defaults and same
#: env-var overrides as 03, so the two notebooks address one cluster.
#: THE PRIMARY IS `-1`, NOT `-0`. Earlier manual resets hit a read-only replica
#: and reported success; a DROP INDEX against a replica fails outright, which is
#: the better failure but still the wrong pod.
NS = os.environ.get("POLARIS_K8S_NAMESPACE", "datahub-hynix")
PGPOD = os.environ.get("PGPOD", "benchmarks-postgresql-postgresql-ha-postgresql-1")

SCHEMA = "polaris_schema"
INDEX = "idx_grant_records_grantee"
CASES = ("unauthorized", "authorized", "admin")
STAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

print(f"Postgres : {PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"kubectl  : -n {NS} {PGPOD}")
print(f"stamp    : {STAMP}")


🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
Postgres : 192.168.139.2:5432/polaris
kubectl  : -n datahub-hynix benchmarks-postgresql-postgresql-ha-postgresql-1
stamp    : 20260903-102125


## Flags

`ARM_TOGGLE` is the only destructive switch. Leave it `False` to sweep the
current state alone.


In [2]:
# --- cell 2: flags ----------------------------------------------------------
#: Sweep BOTH index states. False = sweep only the state the cluster is in.
#:
#: FALSE IS A COMPLETE MEASUREMENT, not a truncated one:
#: `idx_grant_records_grantee` is in no schema file here, so index-absent is
#: the stock Polaris schema. True adds the hypothetical remedy and answers
#: "would that index help" instead. Either way the index is restored to the
#: state it was found in.
ARM_TOGGLE = False

#: Plan every pair even when a statement repeats across APIs. Off by default:
#: the pairs are already deduplicated by (SQL, params).
DRY_RUN_FIRST = True

print(f"ARM_TOGGLE     = {ARM_TOGGLE}"
      f"{'  → both states will be swept' if ARM_TOGGLE else '  → current state only'}")
print(f"DRY_RUN_FIRST  = {DRY_RUN_FIRST}")


ARM_TOGGLE     = False  → current state only
DRY_RUN_FIRST  = True


In [3]:
# --- cell 3: helpers --------------------------------------------------------
def sh(cmd, check=True, quiet=False):
    """Run a shell command from HERE, stream it, raise on failure."""
    p = subprocess.run(cmd, shell=True, cwd=str(HERE), capture_output=True, text=True)
    if not quiet:
        if p.stdout:
            print(p.stdout.rstrip())
        if p.stderr:
            print(p.stderr.rstrip(), file=sys.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")
    return p


def psql(sql, quiet=True):
    """One statement through psql -tAc, on the PRIMARY.

    THE SQL IS FLATTENED TO ONE LINE FIRST. `kubectl exec` passes `\n` through
    literally and psql reads a leading backslash-n as a META-COMMAND, failing
    with `invalid command \nSELECT` — so a multi-line query silently becomes a
    psql syntax error rather than a result. Whitespace is nothing to SQL.
    """
    one_line = " ".join(sql.split())
    return sh(
        f"kubectl exec -n {NS} {PGPOD} -- env PGPASSWORD={PG_PASSWORD} "
        f"psql -U {PG_USER} -d {PG_DB} -tAc {shlex.quote(one_line)}",
        quiet=quiet,
    ).stdout.strip()


def index_present():
    return psql(
        f"SELECT count(*) FROM pg_indexes WHERE indexname = '{INDEX}'"
    ) == "1"


def latest_report(case):
    hits = sorted(REPORTS.glob(f"doc-api-sql-matrix-{case}-*.md"))
    return hits[-1] if hits else None


print("helpers defined")


helpers defined


## Preflight

Nothing is planned until every report is found and the live index state is
known. `explain_api_matrix.py` additionally **refuses a report that did not
fully parse**, before touching the database — a short worklist looks exactly
like a complete one.


In [4]:
# --- cell 4: preflight ------------------------------------------------------
START_STATE = "present" if index_present() else "absent"
OTHER_STATE = "absent" if START_STATE == "present" else "present"
print(f"live index state: {START_STATE}  ({INDEX})")

vol = psql(f"SELECT count(*) FROM {SCHEMA}.grant_records")
print(f"grant_records rows: {int(vol):,}")

MATRICES = {}
for case in CASES:
    r = latest_report(case)
    MATRICES[case] = r
    print(f"  {case:<13} {r.name if r else '!! NO REPORT — drive it first'}")

missing = [c for c, r in MATRICES.items() if r is None]
assert not missing, (
    f"no matrix report for {missing}. Drive those cases in 03 first — sweeping "
    "two of three states produces a contrast nobody can read."
)


live index state: absent  (idx_grant_records_grantee)
grant_records rows: 60,815
  unauthorized  doc-api-sql-matrix-unauthorized-20260902-171852.md
  authorized    doc-api-sql-matrix-authorized-20260902-172636.md
  admin         doc-api-sql-matrix-admin-20260903-094137.md


In [5]:
# --- cell 5: dry run — the worklist, touching no database -------------------
if DRY_RUN_FIRST:
    for case in CASES:
        print(f"\n=== {case} ===")
        sh(f"uv run python explain_api_matrix.py --case {case} "
           f"--index-state {START_STATE} "
           f"--matrix {MATRICES[case].relative_to(HERE)} --dry-run")
else:
    print("DRY_RUN_FIRST is off")



=== unauthorized ===
reports/doc-api-sql-matrix-unauthorized-20260902-171852.md
  43 APIs, 390 statement instances
  33 (SQL, params) pairs over 11 distinct SQL texts
  33 replayable, 0 refused

=== authorized ===
reports/doc-api-sql-matrix-authorized-20260902-172636.md
  43 APIs, 460 statement instances
  93 (SQL, params) pairs over 25 distinct SQL texts
  91 replayable, 2 refused
    - grant_records SELECT: no parameters were recorded for this statement
    - events INSERT: no parameters were recorded for this statement

=== admin ===
reports/doc-api-sql-matrix-admin-20260903-094137.md
  43 APIs, 564 statement instances
  138 (SQL, params) pairs over 30 distinct SQL texts
  128 replayable, 10 refused
    - grant_records SELECT: placeholder/parameter mismatch: 13 placeholders, 5 values
    - grant_records SELECT: placeholder/parameter mismatch: 13 placeholders, 5 values
    - grant_records SELECT: placeholder/parameter mismatch: 13 placeholders, 5 values
    - principal_authenticatio

Installed 9 packages in 8ms


**Read the refusals before continuing.** A refused pair is not planned and not
counted, which is the honest behaviour — but three of them are permanent and the
rest are not:

- `parameters were redacted at capture (secret table)` —
  `principal_authentication_data`, 3 pairs in the admin case. Permanent, by
  design.
- anything else — a parser fault. Stop and fix it rather than sweeping around
  it; the 2026-09-03 dry run found six corrupted `grant_records` statements
  this way.


In [6]:
# --- cell 6: sweep the state the cluster is already in ----------------------
for case in CASES:
    print(f"\n=== {case} · index {START_STATE} ===")
    sh(f"uv run python explain_api_matrix.py --case {case} "
       f"--index-state {START_STATE} "
       f"--matrix {MATRICES[case].relative_to(HERE)}")



=== unauthorized · index absent ===
reports/doc-api-sql-matrix-unauthorized-20260902-171852.md
  43 APIs, 390 statement instances
  33 (SQL, params) pairs over 11 distinct SQL texts
  33 replayable, 0 refused


--index-state absent but pg_indexes says present.
  EXPLAIN replays against the database as it IS. Planning this
  report now would staple {live_state} plans onto a drive taken
  in another state, and the document would not say so.


RuntimeError: exit 1: uv run python explain_api_matrix.py --case unauthorized --index-state absent --matrix reports/doc-api-sql-matrix-unauthorized-20260902-171852.md

## The toggle — optional, and NOT the default measurement

`idx_grant_records_grantee` appears in **no schema file in this repo**. It is a
hand-created experimental index, so **index-absent is the stock Polaris schema**
— what a real deployment runs. Sweeping the absent state alone is therefore a
complete finding about shipped Polaris, not half an experiment.

`ARM_TOGGLE = True` additionally creates the index and sweeps the hypothetical
remedy, then removes it again. Use it to answer "would this index help"; leave
it `False` to answer "what does Polaris do as shipped".

`ANALYZE` after any toggle is **not optional**: the planner would otherwise
choose against statistics describing an index state that no longer exists.


In [7]:
# --- cell 7: toggle to the other state --------------------------------------
if not ARM_TOGGLE:
    print(f"ARM_TOGGLE is False — stopping with the index {START_STATE}.")
    print(f"Set it True and re-run to also sweep '{OTHER_STATE}'.")
else:
    print(f"{START_STATE} → {OTHER_STATE}")
    if START_STATE == "present":
        sh("uv run python drop_grantee_index.py")
    else:
        psql(f"CREATE INDEX CONCURRENTLY {INDEX} "
             f"ON {SCHEMA}.grant_records (realm_id, grantee_catalog_id, grantee_id)",
             quiet=False)
    psql(f"ANALYZE {SCHEMA}.grant_records")
    now = "present" if index_present() else "absent"
    assert now == OTHER_STATE, f"toggle did not take: still {now}"
    print(f"index is now {now}, statistics refreshed")


ARM_TOGGLE is False — stopping with the index absent.
Set it True and re-run to also sweep 'present'.


In [8]:
# --- cell 8: sweep the other state ------------------------------------------
if not ARM_TOGGLE:
    print("skipped — ARM_TOGGLE is False")
else:
    for case in CASES:
        print(f"\n=== {case} · index {OTHER_STATE} ===")
        sh(f"uv run python explain_api_matrix.py --case {case} "
           f"--index-state {OTHER_STATE} "
           f"--matrix {MATRICES[case].relative_to(HERE)}")


skipped — ARM_TOGGLE is False


## Restore

Always run this, including after a failure above. Leaving `grant_records`
without its index is not a neutral state — every later measurement on this
cluster would silently be an index-absent one.


In [9]:
# --- cell 9: restore the state the cluster started in ----------------------
if not ARM_TOGGLE:
    print("nothing to restore — the toggle never ran")
else:
    now = "present" if index_present() else "absent"
    if now == START_STATE:
        print(f"already {START_STATE} — nothing to do")
    else:
        if START_STATE == "present":
            psql(f"CREATE INDEX CONCURRENTLY {INDEX} "
                 f"ON {SCHEMA}.grant_records "
                 f"(realm_id, grantee_catalog_id, grantee_id)", quiet=False)
        else:
            sh("uv run python drop_grantee_index.py")
        psql(f"ANALYZE {SCHEMA}.grant_records")
        back = "present" if index_present() else "absent"
        assert back == START_STATE, f"RESTORE FAILED — index is {back}"
        print(f"restored to {START_STATE}, statistics refreshed")


nothing to restore — the toggle never ran


## Results

With `ARM_TOGGLE = False` there are three runs — one per case, all in the state
the cluster is in. With it armed there are six, and each records the live
`pg_indexes` state it was taken in, so a half can never be mislabelled.


In [10]:
# --- cell 10: what was produced --------------------------------------------
import pandas as pd

rows = []
for f in sorted(RUNS.glob("apiexplain-*.json")):
    d = json.loads(f.read_text())
    per = d.get("per_api", {})
    rels = sorted({x for r in per.values() for x in r.get("seq_scanned", [])})
    rows.append({
        "case": d.get("case"),
        "index": d.get("index_state"),
        "APIs": len(per),
        "pairs": d.get("pairs"),
        "seq-scanning APIs": sum(1 for r in per.values() if r.get("seq_scanned")),
        "relations seq-scanned": ", ".join(rels) or "—",
        "grant_records rows": (d.get("volume") or {}).get("grant_records"),
        "run": f.name,
    })

df = pd.DataFrame(rows).sort_values(["case", "index"]) if rows else None
if df is None:
    print("no apiexplain-*.json yet")
else:
    display(df)
    have = {(r["case"], r["index"]) for _, r in df.iterrows()}
    #: Only a MISSING half is worth flagging, and only when both were asked
    #: for. With the toggle unarmed a single state is the intended result, not
    #: an incomplete run -- saying otherwise trains people to ignore the check.
    if ARM_TOGGLE:
        want = {(c, s) for c in CASES for s in ("present", "absent")}
        if want - have:
            print(f"\nMISSING halves: {sorted(want - have)}")
            print("A contrast built from an incomplete set compares fixtures, "
                  "not indexes.")
    else:
        missing = [c for c in CASES if not any(h[0] == c for h in have)]
        if missing:
            print(f"\nno run for: {missing}")


no apiexplain-*.json yet


In [11]:
# --- cell 11: the finding ---------------------------------------------------
#: Two modes, because the question is different in each.
#:
#: ONE STATE (the default): what does Polaris do AS SHIPPED. The index this
#: notebook can toggle is in no schema file in this repo, so absent is the
#: stock configuration and a single-state sweep is a complete answer.
#:
#: BOTH STATES: would that index help. Keyed on the per-API rollup rather than
#: statement text -- `explains` and the matrix's pairs are zipped positionally
#: and only the explains reach the run file, so the JSON alone cannot say which
#: statement a plan belongs to, while `per_api` is keyed by API name in both
#: halves and needs no alignment.
runs = {}
for f in sorted(RUNS.glob("apiexplain-*.json")):
    d = json.loads(f.read_text())
    runs.setdefault(d.get("case"), {})[d.get("index_state")] = d

both = {c: h for c, h in runs.items() if {"present", "absent"} <= set(h)}

if both:
    rows = []
    for case, halves in sorted(both.items()):
        pres, absent = halves["present"], halves["absent"]
        for api in sorted(set(pres["per_api"]) | set(absent["per_api"])):
            p_row, a_row = pres["per_api"].get(api, {}), absent["per_api"].get(api, {})
            rows.append({
                "case": case, "api": api,
                "present → seq": ", ".join(p_row.get("seq_scanned") or []) or "—",
                "absent → seq": ", ".join(a_row.get("seq_scanned") or []) or "—",
                "changed": sorted(p_row.get("seq_scanned") or [])
                           != sorted(a_row.get("seq_scanned") or []),
            })
    cdf = pd.DataFrame(rows)
    changed = cdf[cdf["changed"]]
    print(f"{len(changed)} of {len(cdf)} (case, API) pairs change scan behaviour "
          f"with the index\n")
    display(changed if len(changed) else cdf.head(20))
    if not len(changed):
        print("NOTHING CHANGED. Check the volume stamped on the runs before "
              "reading that as 'the index does not matter' — at low row counts "
              "the primary key covers the grantee lookup and the toggle is a "
              "no-op by construction.")
else:
    #: Single-state analysis: the stock schema, stated as a finding rather than
    #: as a contrast that failed to materialise.
    rows = []
    for case, halves in sorted(runs.items()):
        state, d = next(iter(halves.items()))
        per = d["per_api"]
        for api, r in sorted(per.items()):
            rows.append({
                "case": case,
                "api": api,
                "seq-scans": ", ".join(r.get("seq_scanned") or []) or "—",
                "indexes used": ", ".join(r.get("indexes_used") or []) or "—",
                "index-only": r.get("uses_index_only"),
                "planned": r.get("planned"),
                "skipped": r.get("skipped"),
            })
    if not rows:
        print("no runs yet")
    else:
        sdf = pd.DataFrame(rows)
        state = next(iter(next(iter(runs.values())).keys()))
        vol = (next(iter(next(iter(runs.values())).values()))
               .get("volume", {}).get("grant_records"))
        seq = sdf[sdf["seq-scans"] != "—"]
        print(f"index {state} — the stock schema — at grant_records = {vol:,} rows"
              if isinstance(vol, int) else f"index {state} — the stock schema")
        print(f"{len(seq)} of {len(sdf)} (case, API) pairs sequentially scan "
              f"something\n")
        display(seq if len(seq) else sdf.head(20))
        hot = sdf[sdf["seq-scans"].str.contains("grant_records")]
        print(f"\n{len(hot)} of {len(sdf)} scan grant_records sequentially — "
              "the authorization prelude every request pays.")


no case has both halves yet — run with ARM_TOGGLE = True


In [12]:
# --- cell 12: teardown ------------------------------------------------------
#: Notebooks here close what they opened. This one shells out rather than
#: holding a pool, so there is nothing to dispose — but the dataframes are
#: dropped explicitly, per the repo convention.
for _n in ("df", "cdf"):
    if _n in dir():
        del globals()[_n]
gc.collect()
print("done. Index state:", "present" if index_present() else "absent")


done. Index state: absent
